# COG403 Project Notebook


## 1. Imports

In [1]:
import json
import math
import random
import re
from collections import Counter
from functools import lru_cache
from itertools import product
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import pandas as pd
import matplotlib.pyplot as plt


## 2. Installations

In [2]:
# Uncomment the first time
# %pip install pronouncing cmudict
# %pip install g2p-en


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 14.6 MB/s eta 0:00:00
  Created wheel for pronouncing: filename=pronouncing-0.2.0-py2.py3-none-any.whl size=6234 sha256=08987ab33ec39469f114d9a2b03409b77f519d3c00568d620979044803b777da
  Stored in directory: /root/.cache/pip/wheels/a0/76/15/dfdf38731993cdc4e86fd6d949c70c0e9786cf00073d8114d4
Successfully built pronouncing
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.3 MB/s eta 0:00:00
  Created wheel for distance: filename=Distance-0.1.3-py3-none-any.whl size=16256 sha256=9374e9f91479f484f511a3455aaaee3ceecbc9352717c11ddc62ac0826310f0d
  Stored in directory: /root/.cache/pip/wheels/24/a8/58/407063d8e5c1d4dd6594c99d12baa0108570b56a92325587dd
Successfully built distance


## 3. Parse Bernstein-Ratner `.cha` files and keep caregiver speech

In [3]:
ADULT_SPEAKERS = {"MOT", "NAN"}

# Kept only for optional robustness checks. The main pipeline retains these items.
INTERJECTION_WORDS = {
    "oh", "o", "o:h", "ah", "uh", "um", "uhh", "umm", "hm", "huh",
    "yeah", "okay", "ok", "well", "hello", "no", "yes"
}


def clean_chat_text(text: str) -> str:
    text = text.strip()

    text = re.sub(r'\+\"[^\s]*', ' ', text)
    text = re.sub(r'\+\.\.\.', ' ', text)
    text = re.sub(r'\+//\.?', ' ', text)

    text = re.sub(r'\[[^\]]*\]', ' ', text)

    text = text.replace('<', ' ').replace('>', ' ')

    text = re.sub(r'\(\.\)', ' ', text)
    text = re.sub(r'\([^\w]*\)', ' ', text)

    text = text.replace('(', '').replace(')', '')

    text = re.sub(r'&[~=\-]?[A-Za-z:]+', ' ', text)
    text = re.sub(r'&=\S+', ' ', text)

    text = re.sub(r'\b(?:xxx|www)\b', ' ', text)

    text = text.replace('a_lot_of', 'a lot of')

    text = re.sub(r"[^A-Za-z'\s]", ' ', text)

    text = text.replace('’', "'").lower()

    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize_cleaned_text(text: str) -> List[str]:
    if not text:
        return []
    return [tok for tok in text.split() if tok]


def remove_interjections(words: List[str], remove_set: Optional[Set[str]] = None) -> List[str]:
    if remove_set is None:
        remove_set = INTERJECTION_WORDS
    return [w for w in words if w not in remove_set]


def extract_adult_utterances_from_cha(path: str, adult_speakers: Optional[Set[str]] = None) -> List[Dict[str, object]]:
    if adult_speakers is None:
        adult_speakers = ADULT_SPEAKERS

    utterances = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.startswith('*'):
                continue

            m = re.match(r'^\*(\w+):\s*(.*)$', line)
            if not m:
                continue

            speaker, content = m.group(1), m.group(2)
            if speaker not in adult_speakers:
                continue

            cleaned = clean_chat_text(content)
            words = tokenize_cleaned_text(cleaned)

            utterances.append({
                'file': str(path),
                'speaker': speaker,
                'raw': content,
                'cleaned': cleaned,
                'words': words,
            })

    return utterances


def parse_all_cha_files(cha_paths: List[str], drop_interjections: bool = False) -> pd.DataFrame:
    rows = []
    for path in cha_paths:
        utts = extract_adult_utterances_from_cha(path)
        for utt in utts:
            words = utt['words']
            if drop_interjections:
                words = remove_interjections(words)
            utt = dict(utt)
            utt['words'] = words
            utt['n_words'] = len(words)
            rows.append(utt)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    df = df[df['n_words'] > 0].reset_index(drop=True)
    return df


## 4. Transcript paths

In [4]:
sample_cha_files = [
    '/content/Alice/010100.cha',
    '/content/Alice/010300.cha',
    '/content/Alice/010500.cha',
    '/content/Amelia/010600.cha',
    '/content/Amelia/010800.cha',
    '/content/Amelia/011000.cha',
    '/content/Anne/010500.cha',
    '/content/Anne/010800.cha',
    '/content/Anne/011000.cha',
    '/content/Cindy/010800.cha',
    '/content/Cindy/011000.cha',
    '/content/Cindy/020000.cha',
    '/content/Dale/010500.cha',
    '/content/Dale/010700.cha',
    '/content/Dale/010900.cha',
    '/content/Gail/010900.cha',
    '/content/Gail/011100.cha',
    '/content/Gail/020100.cha',
    '/content/Kay/010100.cha',
    '/content/Kay/010400.cha',
    '/content/Kay/010600.cha',
    '/content/Lena/010700.cha',
    '/content/Lena/011000.cha',
    '/content/Lena/020000.cha',
    '/content/Marie/010600.cha',
    '/content/Marie/010800.cha',
    '/content/Marie/011000.cha'
]

sample_df = parse_all_cha_files(sample_cha_files, drop_interjections=False)
sample_df[['file', 'speaker', 'raw', 'cleaned', 'words']].head(20)


,file,speaker,raw,cleaned,words
0,/content/Alice/010100.cha,MOT,she's really into books right now .,she's really into books right now,"[she's, really, into, books, right, now]"
1,/content/Alice/010100.cha,MOT,you wanna see the book ?,you wanna see the book,"[you, wanna, see, the, book]"
2,/content/Alice/010100.cha,MOT,"oh , look there's a boy with his hat .",oh look there's a boy with his hat,"[oh, look, there's, a, boy, with, his, hat]"
3,/content/Alice/010100.cha,MOT,and a doggie .,and a doggie,"[and, a, doggie]"
4,/content/Alice/010100.cha,MOT,"oh , you wanna look at this ?",oh you wanna look at this,"[oh, you, wanna, look, at, this]"
5,/content/Alice/010100.cha,MOT,&~a:h look at this ?,look at this,"[look, at, this]"
6,/content/Alice/010100.cha,MOT,have a drink .,have a drink,"[have, a, drink]"
7,/content/Alice/010100.cha,MOT,okay now .,okay now,"[okay, now]"
8,/content/Alice/010100.cha,MOT,oh what's this ?,oh what's this,"[oh, what's, this]"
9,/content/Alice/010100.cha,MOT,what's that ?,what's that,"[what's, that]"


## 5. Pronunciation scaffold: CMUdict pronouncing first, g2p-en fallback

In [5]:
try:
    import pronouncing
    HAS_PRONOUNCING = True
except Exception:
    HAS_PRONOUNCING = False
    print('pronouncing not available; uncomment the pip install cell if needed.')

try:
    from g2p_en import G2p
    HAS_G2P = True
    g2p = G2p()
except Exception:
    HAS_G2P = False
    g2p = None
    print('g2p_en not available; uncomment the pip install cell if needed.')

CMU_VOWELS = {
    "AA","AE","AH","AO","AW","AY","EH","ER","EY","IH","IY","OW","OY","UH","UW"
}

def strip_stress(phones: List[str]) -> List[str]:
    return [re.sub(r'\d', '', p).upper() for p in phones]


def normalize_lookup_word(word: str) -> str:
    return word.lower().strip("'")


def g2p_word_to_phonemes(word: str) -> Optional[List[str]]:
    if not HAS_G2P:
        return None
    try:
        raw = g2p(word)
    except Exception:
        return None

    # Keep only phone-like symbols, remove spaces and punctuation artifacts.
    phones = []
    for tok in raw:
        if not tok or tok == ' ':
            continue
        # g2p-en sometimes emits punctuation or apostrophes; skip those.
        if re.fullmatch(r"[^A-Za-z0-9]+", str(tok)):
            continue
        tok = re.sub(r'\d', '', str(tok)).upper()
        if tok:
            phones.append(tok)

    return phones if phones else None


def word_to_phonemes(word: str) -> Optional[List[str]]:
    w = normalize_lookup_word(word)
    if not w:
        return None

    if HAS_PRONOUNCING:
        phones = pronouncing.phones_for_word(w)
        if phones:
            return strip_stress(phones[0].split())

    # Fallback to g2p only when CMUdict/pronouncing does not have the word.
    return g2p_word_to_phonemes(w)


def words_to_phoneme_utterance(words: List[str]) -> Optional[List[List[str]]]:
    utt = []
    for w in words:
        phs = word_to_phonemes(w)
        if phs is None:
            return None
        utt.append(phs)
    return utt


def convert_dataframe_to_phoneme_utterances(df: pd.DataFrame) -> Tuple[List[List[List[str]]], pd.DataFrame]:
    kept = []
    audit_rows = []

    for _, row in df.iterrows():
        words = row['words']
        phon_utt = words_to_phoneme_utterance(words)
        success = phon_utt is not None

        audit_rows.append({
            'file': row['file'],
            'speaker': row['speaker'],
            'cleaned': row['cleaned'],
            'words': words,
            'success': success,
        })

        if success:
            kept.append(phon_utt)

    audit_df = pd.DataFrame(audit_rows)
    return kept, audit_df


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


## 6. Convert pronunciations into naturally listenable chunks


In [6]:
CONSONANTS = {
    "B","CH","D","DH","F","G","HH","JH","K","L","M","N","NG","P","R","S","SH","T","TH","V","W","Y","Z","ZH"
}

LEGAL_ONSETS = {
    (),
    ("B",),("CH",),("D",),("DH",),("F",),("G",),("HH",),("JH",),("K",),("L",),("M",),("N",),("P",),("R",),("S",),("SH",),("T",),("TH",),("V",),("W",),("Y",),("Z",),
    ("P","R"),("P","L"),("B","R"),("B","L"),("T","R"),("D","R"),("K","R"),("K","L"),("G","R"),("G","L"),
    ("F","R"),("F","L"),("TH","R"),("SH","R"),
    ("S","P"),("S","T"),("S","K"),("S","M"),("S","N"),("S","L"),("S","W"),
    ("S","P","R"),("S","T","R"),("S","K","R"),("S","P","L"),("S","K","W"),
}

def is_vowel(phone: str) -> bool:
    return phone in CMU_VOWELS

def syllabify_phones(phones: List[str]) -> List[List[str]]:
    """Simple onset-maximizing syllabifier over CMU-style phones.
    This tends to produce AE | P_AH_L for APPLE, not AE_P | AH_L.
    """
    if not phones:
        return []
    vowel_positions = [i for i,p in enumerate(phones) if is_vowel(p)]
    if not vowel_positions:
        return [phones[:]]

    syllables = []
    start = 0

    for vi, vpos in enumerate(vowel_positions[:-1]):
        next_v = vowel_positions[vi+1]
        between = phones[vpos+1:next_v]

        split = len(between)
        for k in range(len(between)+1):
            onset = tuple(between[k:])
            if onset in LEGAL_ONSETS:
                split = k
                break

        syll_end = vpos + 1 + split
        syllables.append(phones[start:syll_end])
        start = syll_end

    syllables.append(phones[start:])
    return [s for s in syllables if s]

def phones_to_chunk_tokens(phones: List[str]) -> List[str]:
    syllables = syllabify_phones(phones)
    return ['_'.join(syl) for syl in syllables if syl]

def words_to_chunk_utterance(words: List[str]) -> Optional[List[List[str]]]:
    out = []
    for w in words:
        phs = word_to_phonemes(w)
        if phs is None:
            return None
        chunks = phones_to_chunk_tokens(phs)
        if not chunks:
            return None
        out.append(chunks)
    return out

def convert_dataframe_to_chunk_utterances(df: pd.DataFrame) -> Tuple[List[List[List[str]]], pd.DataFrame]:
    kept = []
    rows = []

    for _, row in df.iterrows():
        words = row['words']
        chunk_utt = words_to_chunk_utterance(words)
        success = chunk_utt is not None

        rows.append({
            'file': row['file'],
            'speaker': row['speaker'],
            'cleaned': row['cleaned'],
            'words': words,
            'success': success,
            'chunk_utt': chunk_utt if success else None,
        })
        if success:
            kept.append(chunk_utt)

    return kept, pd.DataFrame(rows)


## 7. Build the chunk-level dataset

In [7]:
chunk_utterances, audit_df = convert_dataframe_to_chunk_utterances(sample_df)

print('Total cleaned adult utterances:', len(sample_df))
print('Successfully converted to chunk utterances:', len(chunk_utterances))
print('Coverage:', round(100 * len(chunk_utterances) / max(len(sample_df), 1), 2), '%')

audit_df[['cleaned', 'words', 'success']].head(20)


Total cleaned adult utterances: 9798
Successfully converted to chunk utterances: 9308
Coverage: 95.0 %


,cleaned,words,success
0,she's really into books right now,"[she's, really, into, books, right, now]",True
1,you wanna see the book,"[you, wanna, see, the, book]",True
2,oh look there's a boy with his hat,"[oh, look, there's, a, boy, with, his, hat]",True
3,and a doggie,"[and, a, doggie]",True
4,oh you wanna look at this,"[oh, you, wanna, look, at, this]",True
5,look at this,"[look, at, this]",True
6,have a drink,"[have, a, drink]",True
7,okay now,"[okay, now]",True
8,oh what's this,"[oh, what's, this]",True
9,what's that,"[what's, that]",True


## 8. Inspect unresolved words

In [8]:
def unresolved_words(df: pd.DataFrame) -> pd.DataFrame:
    counter = Counter()
    for _, row in df.iterrows():
        for w in row['words']:
            if word_to_phonemes(w) is None:
                counter[w] += 1
    out = pd.DataFrame(counter.items(), columns=['word', 'count']).sort_values('count', ascending=False)
    return out.reset_index(drop=True)

unresolved_words(sample_df).head(50)


,word,count
0,peekaboo,66
1,byebye,55
2,uhoh,33
3,doggie's,26
4,color's,18
5,kitties,16
6,nightie,14
7,bunny's,12
8,woa,10
9,pg,8


## 9. Preview chunkification examples

In [9]:
preview_rows = []
for _, row in audit_df[audit_df['success'] == True].head(15).iterrows():
    preview_rows.append({
        'cleaned': row['cleaned'],
        'word_chunks': [' | '.join(word_chunks) for word_chunks in row['chunk_utt']]
    })

pd.DataFrame(preview_rows)


,cleaned,word_chunks
0,she's really into books right now,"[SH_IY_Z, R_IH | L_IY, IH_N | T_UW, B_UH_K_S, ..."
1,you wanna see the book,"[Y_UW, W_AA | N_AH, S_IY, DH_AH, B_UH_K]"
2,oh look there's a boy with his hat,"[OW, L_UH_K, DH_EH_R_Z, AH, B_OY, W_IH_DH, HH_..."
3,and a doggie,"[AH_N_D, AH, D_AO | G_IY]"
4,oh you wanna look at this,"[OW, Y_UW, W_AA | N_AH, L_UH_K, AE_T, DH_IH_S]"
5,look at this,"[L_UH_K, AE_T, DH_IH_S]"
6,have a drink,"[HH_AE_V, AH, D_R_IH_NG_K]"
7,okay now,"[OW | K_EY, N_AW]"
8,oh what's this,"[OW, W_AH_T_S, DH_IH_S]"
9,what's that,"[W_AH_T_S, DH_AE_T]"


## 10. Train / dev / test split


In [10]:
def random_split_utterances(
    utterances: List[List[List[str]]],
    # change here for different split
    train_ratio: float = 0.8,
    dev_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 403,
):
    assert abs(train_ratio + dev_ratio + test_ratio - 1.0) < 1e-9

    utts = list(utterances)
    rng = random.Random(seed)
    rng.shuffle(utts)

    n = len(utts)
    n_train = int(n * train_ratio)
    n_dev = int(n * dev_ratio)

    train = utts[:n_train]
    dev = utts[n_train:n_train+n_dev]
    test = utts[n_train+n_dev:]
    return train, dev, test

train_utterances, dev_utterances, test_utterances = random_split_utterances(chunk_utterances, seed=403)
print(len(train_utterances), len(dev_utterances), len(test_utterances))


7446 930 932


## 11. Save processed dataset

In [11]:
def save_json_dataset(train_utterances, dev_utterances, test_utterances, path='bernstein_chunk_dataset.json'):
    data = {'train': train_utterances, 'dev': dev_utterances, 'test': test_utterances}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
    print(f'Saved dataset to {Path(path).resolve()}')

save_json_dataset(train_utterances, dev_utterances, test_utterances)


Saved dataset to /content/bernstein_chunk_dataset.json


## 12. Generic utilities over chunk tokens

In [12]:
def flatten_utterance(words_as_chunks: List[List[str]]) -> List[str]:
    return [chunk for word in words_as_chunks for chunk in word]

def gold_boundary_positions(words_as_chunks: List[List[str]]) -> Set[int]:
    positions = set()
    idx = 0
    for word in words_as_chunks[:-1]:
        idx += len(word)
        positions.add(idx)
    return positions

def boundaries_to_segments(stream: List[str], boundaries: Set[int]) -> List[Tuple[str, ...]]:
    segments = []
    start = 0
    for b in sorted(boundaries):
        segments.append(tuple(stream[start:b]))
        start = b
    segments.append(tuple(stream[start:]))
    return segments

def segments_to_boundaries(segments: List[Tuple[str, ...]]) -> Set[int]:
    boundaries = set()
    idx = 0
    for seg in segments[:-1]:
        idx += len(seg)
        boundaries.add(idx)
    return boundaries


## 13. Evaluation

In [13]:
def metrics(pred_boundaries: Set[int], gold_boundaries: Set[int]) -> Tuple[float, float, float]:
    tp = len(pred_boundaries & gold_boundaries)
    fp = len(pred_boundaries - gold_boundaries)
    fn = len(gold_boundaries - pred_boundaries)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def evaluate_segmenter(segmenter, test_utterances: List[List[List[str]]]) -> Dict[str, float]:
    precisions, recalls, f1s = [], [], []
    for utt in test_utterances:
        stream = flatten_utterance(utt)
        gold = gold_boundary_positions(utt)
        pred_segments = segmenter.segment(stream)
        pred = segments_to_boundaries(pred_segments)
        p, r, f1 = metrics(pred, gold)
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

    return {
        'precision': sum(precisions) / len(precisions) if precisions else 0.0,
        'recall': sum(recalls) / len(recalls) if recalls else 0.0,
        'f1': sum(f1s) / len(f1s) if f1s else 0.0,
    }


## Additional evaluation diagnostics

These helpers add over-segmentation vs under-segmentation summaries and MDL lexicon-oriented diagnostics.


In [14]:
from collections import Counter

def segmentation_diagnostics(model, utterances: List[List[List[str]]]) -> pd.DataFrame:
    rows = []
    for utt in utterances:
        stream = flatten_utterance(utt)
        gold = gold_boundary_positions(utt)
        pred_segments = model.segment(stream)
        pred = segments_to_boundaries(pred_segments)

        tp = len(pred & gold)
        fp = len(pred - gold)
        fn = len(gold - pred)

        rows.append({
            'gold_boundaries': len(gold),
            'pred_boundaries': len(pred),
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'over_segmented': max(len(pred) - len(gold), 0),
            'under_segmented': max(len(gold) - len(pred), 0),
        })
    return pd.DataFrame(rows)


def summarize_segmentation_diagnostics(model, utterances: List[List[List[str]]], model_name: str) -> pd.DataFrame:
    diag = segmentation_diagnostics(model, utterances)
    return pd.DataFrame([{
        'Model': model_name,
        'avg_gold_boundaries': diag['gold_boundaries'].mean() if len(diag) else 0.0,
        'avg_pred_boundaries': diag['pred_boundaries'].mean() if len(diag) else 0.0,
        'total_false_positives': int(diag['fp'].sum()) if len(diag) else 0,
        'total_false_negatives': int(diag['fn'].sum()) if len(diag) else 0,
        'avg_over_segmentation': diag['over_segmented'].mean() if len(diag) else 0.0,
        'avg_under_segmentation': diag['under_segmented'].mean() if len(diag) else 0.0,
    }])


def mdl_lexicon_diagnostics(mdl_model, utterances: List[List[List[str]]]) -> pd.DataFrame:
    token_counts = Counter()
    chunk_lengths = []

    for utt in utterances:
        stream = flatten_utterance(utt)
        seg = mdl_model.segment(stream)
        token_counts.update(seg)
        chunk_lengths.extend(len(tok) for tok in seg)

    total_tokens = sum(token_counts.values())
    distinct_types = len(token_counts)
    recurring_token_total = sum(c for c in token_counts.values() if c > 1)
    avg_chunk_len = sum(chunk_lengths) / len(chunk_lengths) if chunk_lengths else 0.0

    return pd.DataFrame([{
        'Model': 'MDL',
        'distinct_chunk_types': distinct_types,
        'total_chunk_tokens': total_tokens,
        'avg_chunk_length': avg_chunk_len,
        'recurring_token_coverage': recurring_token_total / total_tokens if total_tokens else 0.0,
    }])


## 14. TP baseline


In [19]:
class TPSegmenter:
    def __init__(self):
        self.bigram_counts = Counter()
        self.unigram_counts = Counter()
        self.threshold = 0.15

    def fit(self, train_utterances: List[List[List[str]]], dev_utterances=None):
        for utt in train_utterances:
            stream = flatten_utterance(utt)
            for i in range(len(stream)-1):
                x, y = stream[i], stream[i+1]
                self.unigram_counts[x] += 1
                self.bigram_counts[(x, y)] += 1
            if stream:
                self.unigram_counts[stream[-1]] += 1

        if dev_utterances is not None and len(dev_utterances) > 0:
            self.threshold = self.tune_threshold(dev_utterances)
            print(f"Tuned TP threshold: {self.threshold:.2f}")

    def tp(self, x: str, y: str) -> float:
        denom = self.unigram_counts[x]
        return self.bigram_counts[(x, y)] / denom if denom > 0 else 0.0

    def stream_tps(self, stream: List[str]) -> List[float]:
        return [self.tp(stream[i], stream[i+1]) for i in range(len(stream)-1)]

    def local_minima_boundaries(self, stream: List[str], threshold: float) -> Set[int]:
        if len(stream) <= 1:
            return set()
        tps = self.stream_tps(stream)
        boundaries = set()
        for i, val in enumerate(tps):
            if len(tps) == 1:
                local_min = (val < threshold)
            elif i == 0:
                local_min = (val < tps[i+1])
            elif i == len(tps)-1:
                local_min = (val < tps[i-1])
            else:
                local_min = (val < tps[i-1] and val < tps[i+1])
            if local_min or val < threshold:
                boundaries.add(i+1)
        return boundaries

    def tune_threshold(self, dev_utterances: List[List[List[str]]]) -> float:
        best_thr, best_f1 = 0.15, -1.0
        for thr in [i/100 for i in range(1, 51)]:
            f1s = []
            for utt in dev_utterances:
                stream = flatten_utterance(utt)
                gold = gold_boundary_positions(utt)
                pred = self.local_minima_boundaries(stream, thr)
                _, _, f1 = metrics(pred, gold)
                f1s.append(f1)
            avg_f1 = sum(f1s) / len(f1s) if f1s else 0.0
            if avg_f1 > best_f1:
                best_f1, best_thr = avg_f1, thr
        return best_thr

    def segment(self, stream: List[str]) -> List[Tuple[str, ...]]:
        boundaries = self.local_minima_boundaries(stream, self.threshold)
        return boundaries_to_segments(stream, boundaries)


## 15. MDL model

In [16]:
class MDLSegmenter:
    def __init__(
        self,
        lambda_lexicon: float = 5.0,
        max_chunk_len: int = 4,
        min_candidate_freq: int = 2,
        verbose: bool = True,
    ):
        self.lambda_lexicon = lambda_lexicon
        self.max_chunk_len = max_chunk_len
        self.min_candidate_freq = min_candidate_freq
        self.verbose = verbose

        self.global_candidate_counts = Counter()
        self.global_candidate_pool = []
        self.global_candidate_df = None
        self.chunk_inventory = set()
        self.sigma_size = 0

    def fit(self, train_utterances: List[List[List[str]]]):
        streams = [flatten_utterance(utt) for utt in train_utterances]
        self.chunk_inventory = set(ch for s in streams for ch in s)
        self.sigma_size = max(len(self.chunk_inventory), 1)

        candidate_counts = self.collect_candidates(
            streams,
            min_len=1,
            max_len=self.max_chunk_len
        )

        # keep only candidates that appear at least min_candidate_freq times
        if self.min_candidate_freq > 1:
            candidate_counts = Counter({
                cand: freq
                for cand, freq in candidate_counts.items()
                if freq >= self.min_candidate_freq
            })

        self.global_candidate_counts = candidate_counts
        self.global_candidate_pool = candidate_counts.most_common()

        self.global_candidate_df = pd.DataFrame([
            {
                "rank": i,
                "candidate": " | ".join(cand),
                "frequency": freq
            }
            for i, (cand, freq) in enumerate(self.global_candidate_pool, start=1)
        ])

        if self.verbose:
            print(f"Global candidate pool size: {len(self.global_candidate_pool)}")
            with pd.option_context(
                "display.max_rows", None,
                "display.max_colwidth", None,
                "display.expand_frame_repr", False
            ):
                display(self.global_candidate_df)

    def collect_candidates(self, streams: List[List[str]], min_len=1, max_len=4) -> Counter:
        candidates = Counter()
        for stream in streams:
            n = len(stream)
            for i in range(n):
                for L in range(min_len, max_len + 1):
                    end = i + L
                    if end <= n:
                        candidates[tuple(stream[i:end])] += 1
        return candidates

    def lexicon_cost(self, lexicon: Set[Tuple[str, ...]]) -> float:
        # L(lexicon) = sum_{w in V} (lambda + |w| log |Sigma|)
        return sum(
            self.lambda_lexicon + len(w) * math.log(self.sigma_size)
            for w in lexicon
        )

    def corpus_given_lexicon_cost(self, lexicon: Set[Tuple[str, ...]]) -> float:
        # L(corpus | lexicon) = sum_{w in V} c(w) * (-log p(w))
        # where c(w) is corpus-wide frequency and
        # p(w) = c(w) / sum_{u in V} c(u)
        counts = {w: self.global_candidate_counts.get(w, 0) for w in lexicon}

        if any(c == 0 for c in counts.values()):
            return float("inf")

        total = sum(counts.values())
        if total <= 0:
            return float("inf")

        cost = 0.0
        for w, c_w in counts.items():
            p_w = c_w / total
            cost += c_w * (-math.log(p_w))
        return cost

    def segmentation_cost(self, segmentation: List[Tuple[str, ...]]) -> float:
        lexicon = set(segmentation)
        lex_cost = self.lexicon_cost(lexicon)
        corpus_cost = self.corpus_given_lexicon_cost(lexicon)
        return lex_cost + corpus_cost

    def all_segmentations(self, stream: List[str]) -> List[List[Tuple[str, ...]]]:
        n = len(stream)

        @lru_cache(maxsize=None)
        def helper(start: int):
            if start == n:
                return [()]
            out = []
            for L in range(1, self.max_chunk_len + 1):
                end = start + L
                if end <= n:
                    tok = tuple(stream[start:end])

                    # only allow chunks retained in the global candidate pool
                    if tok not in self.global_candidate_counts:
                        continue

                    suffixes = helper(end)
                    for suf in suffixes:
                        out.append((tok,) + suf)
            return out

        return [list(seg) for seg in helper(0)]

    def best_segmentation_with_cost(self, stream: List[str]) -> Tuple[List[Tuple[str, ...]], float]:
        all_segs = self.all_segmentations(stream)

        if not all_segs:
            return [tuple(stream)], float("inf")

        best_seg = None
        best_cost = float("inf")
        for seg in all_segs:
            cost = self.segmentation_cost(seg)
            if cost < best_cost:
                best_cost = cost
                best_seg = seg

        return best_seg, best_cost

    def segment(self, stream: List[str]) -> List[Tuple[str, ...]]:
        best_seg, _ = self.best_segmentation_with_cost(stream)
        return best_seg

## 16. Optional MDL tuning on the dev split



In [17]:
def tune_mdl(
    train_utterances,
    dev_utterances,
    mdl_param_grid: Optional[Dict[str, List]] = None,
):
    if mdl_param_grid is None:
        mdl_param_grid = {
            "lambda_lexicon": [round(x * 0.1, 1) for x in range(1, 101)],  # 0.1 to 10.0
            "max_chunk_len": list(range(2, 11)),                           # 2 to 10
            "min_candidate_freq": list(range(1, 101)),                     # 1 to 100
        } # takes too long, will just use the param grid below

    results = []
    best_params = None
    best_f1 = -1.0

    for lambda_val in mdl_param_grid["lambda_lexicon"]:
        for max_len in mdl_param_grid["max_chunk_len"]:
            for min_freq in mdl_param_grid["min_candidate_freq"]:
                params = {
                    "lambda_lexicon": lambda_val,
                    "max_chunk_len": max_len,
                    "min_candidate_freq": min_freq,
                }

                mdl_model = MDLSegmenter(**params, verbose=False)
                mdl_model.fit(train_utterances)
                dev_scores = evaluate_segmenter(mdl_model, dev_utterances)

                row = {
                    **params,
                    "precision": dev_scores["precision"],
                    "recall": dev_scores["recall"],
                    "f1": dev_scores["f1"],
                }
                results.append(row)

                if dev_scores["f1"] > best_f1:
                    best_f1 = dev_scores["f1"]
                    best_params = params

    results_df = pd.DataFrame(results).sort_values(
        by=["f1", "precision", "recall"],
        ascending=False
    ).reset_index(drop=True)

    return best_params, results_df


mdl_param_grid = {
    "lambda_lexicon": [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "max_chunk_len": [2, 3, 4, 5, 6, 8, 10],
    "min_candidate_freq": [1, 2, 3, 5, 10, 20, 50, 100],
}

# Uncomment to tune:
# best_mdl_params, mdl_dev_results = tune_mdl(train_utterances, dev_utterances, mdl_param_grid)
# mdl_dev_results.head(10)
# print("Best MDL params:", best_mdl_params)


Best MDL params: {'lambda_lexicon': 0.1, 'max_chunk_len': 2, 'min_candidate_freq': 5}


## 17. Run the experiment

In [21]:
def run_experiment(
    train_utterances,
    dev_utterances,
    test_utterances,
    tune_mdl_on_dev: bool = True,
    mdl_param_grid: Optional[Dict[str, List]] = None,
    mdl_params: Optional[Dict] = None,
):
    print("Running TP baseline...")
    tp_model = TPSegmenter()
    tp_model.fit(train_utterances, dev_utterances)
    tp_results = evaluate_segmenter(tp_model, test_utterances)

    if tune_mdl_on_dev:
        if mdl_param_grid is None:
            mdl_param_grid = {
                "lambda_lexicon": [round(x * 0.1, 1) for x in range(1, 101)],  # 0.1 to 10.0
                "max_chunk_len": list(range(2, 11)),                           # 2 to 10
                "min_candidate_freq": list(range(1, 101)),                     # 1 to 100
            }

        print("Tuning MDL on dev split...")
        best_mdl_params, mdl_dev_results = tune_mdl(
            train_utterances,
            dev_utterances,
            mdl_param_grid
        )
    else:
        best_mdl_params = mdl_params or {
            'lambda_lexicon': 5.0,
            'max_chunk_len': 4,
            'min_candidate_freq': 2,
        }
        mdl_dev_results = None

    print("Chosen MDL params:", best_mdl_params)
    mdl_model = MDLSegmenter(**best_mdl_params, verbose=True)
    mdl_model.fit(train_utterances)
    mdl_results = evaluate_segmenter(mdl_model, test_utterances)

    results_df = pd.DataFrame([
        {'Model': 'TP', **tp_results},
        {'Model': 'MDL', **mdl_results},
    ])

    return tp_model, mdl_model, results_df, best_mdl_params, mdl_dev_results


tp_model, mdl_model, results_df, best_mdl_params, mdl_dev_results = run_experiment(
    train_utterances,
    dev_utterances,
    test_utterances,
    tune_mdl_on_dev=False,
    mdl_params={ # adjust MDL hyperparams here
        'lambda_lexicon': 0.1,
        'max_chunk_len': 3,
        'min_candidate_freq': 5,
    }
)

results_df

Running TP baseline...
Tuned TP threshold: 0.49
Chosen MDL params: {'lambda_lexicon': 0.1, 'max_chunk_len': 3, 'min_candidate_freq': 5}
Global candidate pool size: 2034


,rank,candidate,frequency
0,1,Y_UW,1074
1,2,DH_AH,1002
2,3,AH,975
3,4,OW,947
4,5,DH_AE_T,632
5,6,W_AH_T,613
6,7,IH_Z,557
7,8,IH_T,508
8,9,DH_IH_S,470
9,10,D_UW,459


,Model,precision,recall,f1
0,TP,0.711338,0.760841,0.727730
1,MDL,0.479026,0.311024,0.364056


## Diagnostic evaluations


In [22]:
tp_diag_summary = summarize_segmentation_diagnostics(tp_model, test_utterances, 'TP')
mdl_diag_summary = summarize_segmentation_diagnostics(mdl_model, test_utterances, 'MDL')
diagnostic_summary_df = pd.concat([tp_diag_summary, mdl_diag_summary], ignore_index=True)
diagnostic_summary_df


,Model,avg_gold_boundaries,avg_pred_boundaries,total_false_positives,total_false_negatives,avg_over_segmentation,avg_under_segmentation
0,TP,2.726395,3.005365,326,66,0.334764,0.055794
1,MDL,2.726395,1.178112,83,1526,0.015021,1.563305


## MDL lexicon-oriented diagnostics


In [23]:
mdl_lexicon_diag_df = mdl_lexicon_diagnostics(mdl_model, test_utterances)
mdl_lexicon_diag_df


,Model,distinct_chunk_types,total_chunk_tokens,avg_chunk_length,recurring_token_coverage
0,MDL,946,2030,1.984729,0.710345


## 18. Preview predicted segmentations

In [24]:
def preview_segmentations(model, utterances: List[List[List[str]]], n: int = 10) -> pd.DataFrame:
    rows = []
    for utt in utterances[:n]:
        stream = flatten_utterance(utt)
        gold = ' || '.join(' | '.join(word_chunks) for word_chunks in utt)
        pred = ' || '.join(' | '.join(seg) for seg in model.segment(stream))
        rows.append({'gold': gold, 'predicted': pred})
    return pd.DataFrame(rows)

# change values to inspect more segmentations
tp_preview = preview_segmentations(tp_model, test_utterances, n=10)
mdl_preview = preview_segmentations(mdl_model, test_utterances, n=10)

print("TP preview")
display(tp_preview)
print("MDL preview")
display(mdl_preview)


TP preview


,gold,predicted
0,IH_Z || IH_T || M_AA | M_IY,IH_Z || IH_T || M_AA | M_IY
1,D_UW || Y_AA || L_AY_K || HH_IH_M,D_UW || Y_AA || L_AY_K || HH_IH_M
2,DH_AE_T_S || R_AY_T,DH_AE_T_S || R_AY_T
3,W_AA_N_T || T_UW || AH_N | T_AY || M_AY || SH_UW,W_AA_N_T || T_UW || AH_N || T_AY || M_AY || SH_UW
4,W_EH_R_Z || Y_AO_R || N_OW_Z,W_EH_R_Z || Y_AO_R || N_OW_Z
5,G_UH_D || G_ER_L,G_UH_D || G_ER_L
6,N_OW,N_OW
7,D_IY || EH_L,D_IY || EH_L
8,K_AE_N || Y_UW || IH_T,K_AE_N || Y_UW || IH_T
9,HH_M,HH_M


MDL preview


,gold,predicted
0,IH_Z || IH_T || M_AA | M_IY,IH_Z | IH_T || M_AA | M_IY
1,D_UW || Y_AA || L_AY_K || HH_IH_M,D_UW | Y_AA | L_AY_K || HH_IH_M
2,DH_AE_T_S || R_AY_T,DH_AE_T_S | R_AY_T
3,W_AA_N_T || T_UW || AH_N | T_AY || M_AY || SH_UW,W_AA_N_T || T_UW || AH_N | T_AY || M_AY || SH_UW
4,W_EH_R_Z || Y_AO_R || N_OW_Z,W_EH_R_Z | Y_AO_R || N_OW_Z
5,G_UH_D || G_ER_L,G_UH_D | G_ER_L
6,N_OW,N_OW
7,D_IY || EH_L,D_IY | EH_L
8,K_AE_N || Y_UW || IH_T,K_AE_N | Y_UW || IH_T
9,HH_M,HH_M
